# Lecture 14: Advanced Transformer Topics — Answer Key
### NLP Course 2027 — Week 07

---

> **Instructor Answer Key** — Complete worked solutions for all exercises.

**Topics covered:** Cross-lingual transfer, quantization, LoRA/PEFT, sliding window inference.

In [ ]:
# Setup and imports
import os
import time
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline,
)
from sklearn.datasets import load_iris  # not used, just verifying sklearn

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')

---

## Exercise 14.1 — Cross-Lingual NER with XLM-RoBERTa

**Task:** Load a multilingual NER model and run it on the same sentence (about Barack Obama) in 5 languages. Compare entity detection per language.

**Key concept:** XLM-RoBERTa is trained on 100 languages with a shared multilingual vocabulary (250K tokens using SentencePiece BPE). Its shared representation space allows **zero-shot cross-lingual transfer**: entities learned from English labels transfer to French, Spanish, German, Italian without any re-training.

Expected entities per sentence:
- **PER** (person): `Barack Obama` — highly consistent across languages since it is a proper name.
- **LOC** (location): `Hawaii` — consistent; proper place name.
- **ORG / MISC**: `United States / États-Unis / Estados Unidos / USA / Stati Uniti` — may vary by language surface form.

In [ ]:
# Exercise 14.1 — Answer
# Model: Babelscape/wikineural-multilingual-ner (fine-tuned XLM-R on WikiNEural)
# Alternatively: xlm-roberta-large-finetuned-conll03-english (EN-focused)

multilingual_sentences = {
    'English': 'Barack Obama was born in Hawaii and served as President of the United States.',
    'French':  'Barack Obama est né à Hawaii et a servi comme Président des États-Unis.',
    'Spanish': 'Barack Obama nació en Hawaii y sirvió como Presidente de los Estados Unidos.',
    'German':  'Barack Obama wurde in Hawaii geboren und diente als Präsident der USA.',
    'Italian': 'Barack Obama è nato alle Hawaii e ha servito come Presidente degli Stati Uniti.',
}

# Load multilingual NER pipeline
ner_pipe = pipeline(
    'ner',
    model='Babelscape/wikineural-multilingual-ner',
    aggregation_strategy='simple',  # merges subword tokens into full words
)

print(f'{'Language':<12} {'Entity':<20} {'Type':<8} {'Score':<8}')
print('-' * 52)

results_by_lang = {}
for lang, sentence in multilingual_sentences.items():
    entities = ner_pipe(sentence)
    results_by_lang[lang] = entities
    for ent in entities:
        print(f'{lang:<12} {ent["word"]:<20} {ent["entity_group"]:<8} {ent["score"]:.3f}')
    if not entities:
        print(f'{lang:<12} (no entities detected)')

# Summary: which entity types transfer best?
print('\n--- Transfer Analysis ---')
entity_type_counts = {}
for lang, ents in results_by_lang.items():
    for e in ents:
        et = e['entity_group']
        entity_type_counts[et] = entity_type_counts.get(et, 0) + 1

for etype, count in sorted(entity_type_counts.items(), key=lambda x: -x[1]):
    print(f'  {etype}: detected {count} times across 5 languages')

print('\nConclusion: PER (person names) transfer most reliably because')
print('proper names like "Barack Obama" are similar across all languages.')
print('LOC also transfers well. ORG/MISC may vary with surface forms.')

**Why PER transfers best:** Person names are often borrowed across languages (Obama stays Obama). Location names like Hawaii also remain stable. Country names (`United States`, `États-Unis`, `USA`) are the most variable — the model must learn that these are the same entity across languages through multilingual co-training.

**XLM-RoBERTa training:** Trained on 2.5 TB of CommonCrawl data in 100 languages. The shared SentencePiece vocabulary means that subword tokens often overlap across closely related languages (EN/FR/ES/IT share many roots), enabling strong cross-lingual alignment.

---

## Exercise 14.2 — Dynamic Quantization of DistilBERT

**Task:** Apply INT8 dynamic quantization. Measure (1) model size reduction on disk, (2) CPU inference speedup on 50 texts.

**Key concept:** Dynamic quantization converts `nn.Linear` weights from FP32 (4 bytes/param) to INT8 (1 byte/param) at load time. Activations are quantized dynamically at runtime. No calibration data needed. Typical results:
- Size: ~4× smaller on disk
- Speed: 1.5–2.5× faster on CPU (depends on hardware)
- Accuracy: < 1% degradation on most tasks

In [ ]:
# Exercise 14.2 — Answer
import os
import tempfile

MODEL_NAME = 'distilbert-base-uncased-finetuned-sst-2-english'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

# --- 1. Apply dynamic quantization ---
quantized_model = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},   # quantize all Linear layers
    dtype=torch.qint8
)
quantized_model.eval()

# --- 2. Measure model size on disk ---
with tempfile.TemporaryDirectory() as tmpdir:
    fp32_path = os.path.join(tmpdir, 'model_fp32.pt')
    int8_path  = os.path.join(tmpdir, 'model_int8.pt')

    torch.save(model.state_dict(), fp32_path)
    torch.save(quantized_model.state_dict(), int8_path)

    fp32_size_mb = os.path.getsize(fp32_path) / 1e6
    int8_size_mb  = os.path.getsize(int8_path)  / 1e6

print(f'FP32 model size: {fp32_size_mb:.1f} MB')
print(f'INT8 model size: {int8_size_mb:.1f}  MB')
print(f'Size reduction:  {fp32_size_mb / int8_size_mb:.2f}x smaller')

# --- 3. Benchmark inference speed on 50 texts ---
texts = ['This is a great product!'] * 25 + ['The service was absolutely terrible.'] * 25

def run_inference(mdl, texts):
    """Run all texts through the model, return total elapsed time."""
    latencies = []
    for text in texts:
        inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
        t0 = time.time()
        with torch.no_grad():
            _ = mdl(**inputs)
        latencies.append(time.time() - t0)
    return latencies

# Warm up
_ = run_inference(model, texts[:3])
_ = run_inference(quantized_model, texts[:3])

fp32_lats = run_inference(model, texts)
int8_lats  = run_inference(quantized_model, texts)

fp32_avg = np.mean(fp32_lats) * 1000
int8_avg  = np.mean(int8_lats)  * 1000

print(f'\nFP32 avg latency: {fp32_avg:.1f} ms/text')
print(f'INT8 avg latency: {int8_avg:.1f}  ms/text')
print(f'Speedup:          {fp32_avg / int8_avg:.2f}x')

# Verify accuracy is preserved
label_map = {0: 'NEGATIVE', 1: 'POSITIVE'}
sample = 'This NLP course is fantastic!'
inp = tokenizer(sample, return_tensors='pt')
with torch.no_grad():
    fp32_pred = model(**inp).logits.argmax(-1).item()
    int8_pred  = quantized_model(**inp).logits.argmax(-1).item()
print(f'\nSample: "{sample}"')
print(f'FP32 prediction: {label_map[fp32_pred]}')
print(f'INT8 prediction: {label_map[int8_pred]}')
print(f'Predictions match: {fp32_pred == int8_pred}')

**Why INT8 is faster on CPU:** Modern CPUs (Intel AVX-512, ARM Neon) have SIMD instructions for 8-bit integer arithmetic that process 4× more values per clock cycle than 32-bit floats. The quantization overhead (scale/zero-point lookup) is negligible compared to the matrix multiply speedup.

**Trade-offs:**
| Property | FP32 | INT8 |
|----------|------|------|
| Size | ~268 MB | ~70 MB |
| CPU latency | baseline | 1.5–2.5× faster |
| GPU | preferred | no benefit (FP16 is better on GPU) |
| Accuracy drop | 0% | < 0.5% on SST-2 |

---

## Exercise 14.3 — LoRA Fine-Tuning with PEFT

**Task:** Apply LoRA (r=8, alpha=32, target_modules=['q_lin','v_lin']) to DistilBERT for sentiment. Compare trainable parameter count and accuracy vs full fine-tuning.

**Key concept — LoRA math:**
```
Original weight matrix: W  (shape d × k, frozen)
LoRA adds:              ΔW = A × B  (A: d×r, B: r×k)
Forward pass:           output = x(W + ΔW·α/r)
```
With r=8, a (768×768) attention matrix goes from 589,824 trainable params to 2×(768×8) = 12,288 — a **48× reduction** for that layer.

In [ ]:
# Exercise 14.3 — Answer
try:
    from peft import LoraConfig, get_peft_model, TaskType
    from datasets import load_dataset
    from transformers import TrainingArguments, Trainer
    import evaluate

    # --- Load base model ---
    base_model = AutoModelForSequenceClassification.from_pretrained(
        'distilbert-base-uncased', num_labels=2
    )

    # --- Count full model parameters ---
    total_params = sum(p.numel() for p in base_model.parameters())
    trainable_full = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
    print(f'Full fine-tuning: {trainable_full:,} trainable / {total_params:,} total')

    # --- Apply LoRA ---
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=8,
        lora_alpha=32,        # scaling = lora_alpha / r = 4
        lora_dropout=0.1,
        target_modules=['q_lin', 'v_lin'],  # DistilBERT's query and value projections
        bias='none',
    )
    lora_model = get_peft_model(base_model, lora_config)
    lora_model.print_trainable_parameters()

    trainable_lora = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
    print(f'\nParameter reduction: {trainable_full / trainable_lora:.1f}x fewer trainable params')

    # --- Load SST-2 data (small subset for demo) ---
    dataset = load_dataset('glue', 'sst2')
    tokenizer_distil = AutoTokenizer.from_pretrained('distilbert-base-uncased')

    def tokenize(batch):
        return tokenizer_distil(batch['sentence'], truncation=True, max_length=128, padding='max_length')

    small_train = dataset['train'].select(range(2000))
    small_val   = dataset['validation']
    small_train = small_train.map(tokenize, batched=True)
    small_val   = small_val.map(tokenize, batched=True)
    small_train.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    small_val.set_format('torch',   columns=['input_ids', 'attention_mask', 'label'])

    accuracy_metric = evaluate.load('accuracy')

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return accuracy_metric.compute(predictions=preds, references=labels)

    # --- Train LoRA model ---
    training_args = TrainingArguments(
        output_dir='/tmp/lora_distilbert',
        num_train_epochs=2,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        evaluation_strategy='epoch',
        learning_rate=3e-4,   # LoRA typically uses higher LR than full FT
        logging_steps=50,
        save_strategy='no',
        report_to='none',
    )

    trainer = Trainer(
        model=lora_model,
        args=training_args,
        train_dataset=small_train,
        eval_dataset=small_val,
        compute_metrics=compute_metrics,
    )

    t0 = time.time()
    trainer.train()
    lora_train_time = time.time() - t0
    lora_eval = trainer.evaluate()

    print(f'\nLoRA Results:')
    print(f'  Trainable params: {trainable_lora:,}')
    print(f'  Training time:    {lora_train_time:.0f}s')
    print(f'  Val accuracy:     {lora_eval["eval_accuracy"]:.4f}')
    print(f'\nExpected: LoRA achieves ~91-92% on this subset (vs ~92-93% full FT)')
    print(f'Benefit: {trainable_full/trainable_lora:.0f}x fewer params, comparable accuracy, faster per-step.')

except ImportError as e:
    print(f'Missing dependency: {e}')
    print('Install with: pip install peft datasets evaluate')
    print()
    # Show the key numbers analytically
    d, k, r = 768, 768, 8
    full_params_one_layer = d * k
    lora_params_one_layer = d * r + r * k
    print(f'DistilBERT q_lin / v_lin weight: {d}x{k} = {full_params_one_layer:,} params each')
    print(f'LoRA r=8 replacement:            {d}x{r} + {r}x{k} = {lora_params_one_layer:,} params')
    print(f'Compression ratio per layer:     {full_params_one_layer/lora_params_one_layer:.1f}x')
    print()
    print('DistilBERT has 6 transformer layers, 2 target modules each = 12 adapted layers.')
    frozen = 66_000_000 - 12 * full_params_one_layer
    lora_trainable = 12 * lora_params_one_layer
    print(f'Full fine-tuning trainable: ~66,000,000 params')
    print(f'LoRA trainable:             ~{lora_trainable:,} params ({lora_trainable/66e6*100:.2f}%)')

**LoRA summary:**
- `r=8`, `lora_alpha=32`: scaling factor = 32/8 = 4. Higher alpha → larger updates.
- `target_modules=['q_lin', 'v_lin']`: Only query and value projections are adapted. Key and output are frozen.
- DistilBERT has 6 layers × 2 modules = 12 LoRA pairs × (768×8 + 8×768) = ~150K trainable params vs 66M total.
- **Rule of thumb:** LoRA achieves 95–99% of full fine-tuning accuracy with 0.1–1% of trainable parameters.

---

## Exercise 14.4 — Sliding Window Classification for Long Documents

**Task:** Classify a ~1000-token document using sliding window. Test strides [64, 128, 256, 512]. Compare number of chunks, inference time, and final prediction.

**Key concept:** BERT-family models have a hard 512-token limit. For longer documents, we split into overlapping chunks (stride < window creates overlap), classify each chunk independently, then **aggregate predictions** (average logits or majority vote).

```
Document tokens: [0 ... 999]
Window=512, stride=256:
  Chunk 1: tokens [0..511]
  Chunk 2: tokens [256..767]   ← 256 overlapping tokens
  Chunk 3: tokens [512..1023]  ← but doc only goes to 999
```
Larger stride → fewer chunks → faster inference but less coverage of context boundaries.

In [ ]:
# Exercise 14.4 — Answer
MODEL_NAME = 'distilbert-base-uncased-finetuned-sst-2-english'
tokenizer_sw = AutoTokenizer.from_pretrained(MODEL_NAME)
model_sw = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model_sw.eval()

label_map = {0: 'NEGATIVE', 1: 'POSITIVE'}

def sliding_window_classify(text, model, tokenizer, window=512, stride=256, verbose=False):
    """
    Classify a long document using overlapping windows.

    Args:
        window: max tokens per chunk (including [CLS] and [SEP])
        stride: how many tokens to advance between chunks

    Returns:
        (predicted_label, all_logits, n_chunks, elapsed_ms)
    """
    # Tokenize without special tokens to get raw IDs
    token_ids = tokenizer(text, add_special_tokens=False)['input_ids']
    max_content = window - 2  # reserve 2 spots for [CLS] and [SEP]

    all_logits = []
    chunk_ranges = []

    t0 = time.time()
    start = 0
    while start < len(token_ids):
        end = min(start + max_content, len(token_ids))
        chunk_ids = [tokenizer.cls_token_id] + token_ids[start:end] + [tokenizer.sep_token_id]
        attention_mask = [1] * len(chunk_ids)

        input_tensor = torch.tensor([chunk_ids])
        mask_tensor  = torch.tensor([attention_mask])

        with torch.no_grad():
            logits = model(input_ids=input_tensor, attention_mask=mask_tensor).logits

        all_logits.append(logits)
        chunk_ranges.append((start, end))

        if verbose:
            pred = logits.argmax(-1).item()
            print(f'  Chunk [{start:4d}:{end:4d}] → {label_map[pred]} (logits: {logits.tolist()[0]})')

        if end >= len(token_ids):
            break
        start += stride

    elapsed_ms = (time.time() - t0) * 1000

    # Aggregate: average logits across all chunks
    avg_logits = torch.stack(all_logits).mean(dim=0)
    predicted_label = avg_logits.argmax(-1).item()

    return predicted_label, all_logits, len(all_logits), elapsed_ms


# Build a ~1000-token document with mixed sentiment
# (positive first half, negative second half)
positive_part = 'This movie is absolutely brilliant and captivating. The acting is superb. ' * 15
negative_part = 'However the ending was terrible and deeply disappointing. Worst conclusion ever. ' * 15
long_doc = positive_part + negative_part

# Check actual token count
n_tokens = len(tokenizer_sw(long_doc, add_special_tokens=False)['input_ids'])
print(f'Document length: {n_tokens} tokens ({len(long_doc.split())} words)\n')

# Test across strides
strides = [64, 128, 256, 512]
window = 512

print(f'{'Stride':>8} | {'Chunks':>7} | {'Time (ms)':>10} | {'Prediction'}')
print('-' * 50)

for stride in strides:
    pred, logits_list, n_chunks, elapsed = sliding_window_classify(
        long_doc, model_sw, tokenizer_sw, window=window, stride=stride
    )
    print(f'{stride:>8} | {n_chunks:>7} | {elapsed:>10.1f} | {label_map[pred]}')

print()
print('Analysis:')
print('  stride=64 : most chunks, highest coverage of boundaries, slowest')
print('  stride=512: no overlap (tumbling window), fewest chunks, fastest')
print('  Prediction may vary: the mixed document makes aggregation crucial.')
print('  Averaging logits is more robust than majority vote for borderline cases.')

**Stride trade-off table:**

| Stride | Overlap | # Chunks (1000-tok doc) | Speed | Boundary coverage |
|--------|---------|------------------------|-------|-------------------|
| 64 | 448 tok | ~16 | Slow | Maximum |
| 128 | 384 tok | ~8 | Medium | High |
| 256 | 256 tok | ~4 | Fast | Moderate |
| 512 | 0 tok | ~2 | Fastest | None |

**Production recommendation:** stride = 256 (50% overlap) is a good default. For very long documents (books, contracts), consider hierarchical approaches: summarize each chunk, then classify the summaries.

---

## Summary — Key Concepts

| Exercise | Topic | Key Takeaway |
|----------|-------|--------------|
| 14.1 | Cross-lingual NER | XLM-RoBERTa transfers PER > LOC > ORG across 5 languages; proper names are most stable |
| 14.2 | Dynamic quantization | INT8 gives ~4× size reduction and 1.5–2.5× CPU speedup with < 0.5% accuracy loss |
| 14.3 | LoRA / PEFT | r=8 reduces trainable params by ~48× per layer; achieves 95–99% of full fine-tuning accuracy |
| 14.4 | Sliding window | Stride controls overlap vs speed; average logits across chunks for final prediction |

**General principle:** Choose efficiency technique based on constraint:
- Memory-limited → quantization or distillation
- Label-limited → LoRA / PEFT (few-shot fine-tuning)
- Multi-language → XLM-RoBERTa zero-shot transfer
- Long input → sliding window or hierarchical processing

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**